Απαραίτητα imports και επικοινωνία με το minio προκειμένου να κατεβάσω το αντικείμενο το οποίο θέλω και να το επεξεργαστώ με pandas. Όπως και στο notebook με την google δεν έχει κάποια ιδιαίτερη πολυπλοκότητα. Απλό διάβασμα αντικειμένου και μετατροπή σε pandad df

In [39]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config

client = config.create_minio_client()

[Bucket('aws'), Bucket('azure'), Bucket('google'), Bucket('google-clean')]


In [40]:

object_name = "azure_page_3.json"

try:
    response = client.get_object(config.PROVIDERS.get("azure").get("bucket"), object_name=object_name)

    df = pd.read_json(response)

    response.close()
    response.release_conn()

    print ("Success. Page loaded and converted into dataframe")
    print ("Array size: Rows = ",df.shape[0], " and Columns = ", df.shape[1])


except Exception as e:
    print ("Error: ",e)


Success. Page loaded and converted into dataframe
Array size: Rows =  1000  and Columns =  6


Παίρνω ένα λεξικό με 6 key value pairs εκ των οποίων τα 6 είναι μεταδεδομένα οπότε τα πετάω και κρατάω μόνο το items (key) όπου στην τιμή έχει λίστα 1000 στοιχείων κάθε στοιχείο της οποίας είναι λεξικό που έχει μέσα πληροφορίες για την υπηρεσία που θέλω.  Παρακάτω φαίνεται ο πίνακας στην αρχική του μορφή με τα 6 κλειδιά.

In [41]:
df.head()

,BillingCurrency,CustomerEntityId,CustomerEntityType,Items,NextPageLink,Count
0,USD,Default,Retail,"{'currencyCode': 'USD', 'tierMinimumUnits': 0....",https://prices.azure.com:443/api/retail/prices...,1000
1,USD,Default,Retail,"{'currencyCode': 'USD', 'tierMinimumUnits': 0....",https://prices.azure.com:443/api/retail/prices...,1000
2,USD,Default,Retail,"{'currencyCode': 'USD', 'tierMinimumUnits': 0....",https://prices.azure.com:443/api/retail/prices...,1000
3,USD,Default,Retail,"{'currencyCode': 'USD', 'tierMinimumUnits': 0....",https://prices.azure.com:443/api/retail/prices...,1000
4,USD,Default,Retail,"{'currencyCode': 'USD', 'tierMinimumUnits': 0....",https://prices.azure.com:443/api/retail/prices...,1000


Κάνω normalize βάση του items (η normalize παίρνει ένα dictionary και το ανοίγει σε άλλο πίνακα με νεο index με όλα τα πρώτα επιπέδου key value pair), και πλέον στην μεταβλητή έχω έναν πίνακα με τις υπηρεσίες -μία γραμμή αντιστοιχεί σε υπηρεσία- και στις στήλες έχω τα χαρακτηριστικά των υπηρεσιών (πχ: τιμή, τύπος υπηρεσίας, οικογένεια στην οποία ανήκει η υπηρεσία, γεωγραφικές συντεταγμένες υπηρεσίας και άλλα...)
Στα θετικά νέα είναι ότι παρατηρώντας τον πίνακα δεν χρειάζεται καμία περαιτέρω επεξεργασία όπως είχαμε στην google. Δεν υπάρχουν φωλιασμένα πεδία, ούτε λίστες μέσα σε λίστες και περίεργες δομές. Ο πίνακας είναι καθαρός και έτοιμος για ανάγνωση.
Ένα από τα μεταδομένα που είχε το json μόλις το κατέβασα ήταν το νόμισμα. Ωστόσο το πετάω καθώς ο τύπος του νομίσματος στον οποίο είναι εκφρασμένες οι τιμές, βρίσκεται σε κάθε υπηρεσία με την ονομασία currencyCode. Οπότε να το κρατήσω εις διπλούν θα ήταν άχρηστο. Επίσης από τα μεταδομένα δεν κρατάω το nextPageLink καθώς δεν χρειάζεται στην ανάλυση αλλά μόνο στο ingestion, καθώς και ένα κλειδί count το οποίο δηλώνει πόσες εγγραφές έχω στο dict. Αυτό μπορώ να το δω και με μία απλή καταμέτρηση μέσω pandas, και επιπροσθέτως δεν χρειάζεται.
ΣΧΕΤΙΚΑ ΜΕ ΤΑ CustomerEntityId, και CustomerEntityType τα έχω πετάξει αλλά είναι ακόμη υπό εξέταση η χρησιμότητα τους.

In [42]:
df_flat = pd.json_normalize(df['Items'])
pd.set_option('display.max_columns', None)
df_flat.head(10)


,currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName,reservationTerm
0,USD,0.0,0.505452,0.505452,usgovtexas,US Gov TX,2018-11-01T00:00:00Z,0cab4972-6771-43c3-a756-e62e08ae8e75,D2 v2 AHB,DZH318Z0BQN9,DZH318Z0BQN9/004D,SSIS Standard D-series v2 VM,D2 v2,Azure Data Factory v2,DZH315FBNNW2,Analytics,1 Hour,Consumption,False,,NaN
1,USD,0.0,0.000000,0.000000,austriaeast,AT East,2025-05-01T00:00:00Z,ba97cdcf-f753-5722-9a3b-cd0f7130c423,Delete Operations,DZH318Z0BP09,DZH318Z0BP09/01D5,Files v2,Standard LRS,Storage,DZH317F1HKN0,Storage,10K,Consumption,False,,NaN
2,USD,0.0,0.037500,0.037500,uksouth,UK South,2023-05-01T00:00:00Z,0082698b-72e6-5709-8e5c-9e1553f91378,Cold ZRS Index Tags,DZH318Z0BPH7,DZH318Z0BPH7/02JK,General Block Blob v2,Cold ZRS,Storage,DZH317F1HKN0,Storage,10K/Month,Consumption,True,Cold,NaN
3,USD,0.0,0.030000,0.030000,francecentral,FR Central,2021-08-01T00:00:00Z,0a623b39-fbc1-5b4a-a451-bcf16f060fe6,Azure Maps Creator Feature State,DZH318Z0BQ9W,DZH318Z0BQ9W/0034,Azure Maps,Azure Maps Creator,Azure Maps,DZH315B3SJD3,Internet of Things,1K,Consumption,False,,NaN
4,USD,0.0,1.808000,1.808000,westus2,US West 2,2021-11-01T00:00:00Z,0082c735-0257-5cd1-8505-d2b775605910,D32d v5,DZH318Z08MC2,DZH318Z08MC2/004V,Virtual Machines Ddv5 Series,Standard_D32d_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_D32d_v5,NaN
5,USD,0.0,9344.000000,9344.000000,westus2,US West 2,2025-04-01T00:00:00Z,0082c735-0257-5cd1-8505-d2b775605910,D32d v5,DZH318Z08MC2,DZH318Z08MC2/01ZQ,Virtual Machines Ddv5 Series,Standard_D32d_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Reservation,True,Standard_D32d_v5,1 Year
6,USD,0.0,18055.000000,18055.000000,westus2,US West 2,2021-11-01T00:00:00Z,0082c735-0257-5cd1-8505-d2b775605910,D32d v5,DZH318Z08MC2,DZH318Z08MC2/01ZN,Virtual Machines Ddv5 Series,Standard_D32d_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Reservation,True,Standard_D32d_v5,3 Years
7,USD,0.0,3.620417,3.620417,southafricanorth,ZA North,2025-08-01T00:00:00Z,0083388c-4e15-534c-8213-83edad4aa028,FX96-24ms v2 Spot,DZH318Z0H7F2,DZH318Z0H7F2/05V6,Virtual Machines FXmsv2 Series Windows,FX96-24ms v2 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_FX96-24ms_v2,NaN
8,USD,0.0,2.804340,2.804340,southafricanorth,ZA North,2025-08-01T00:00:00Z,0083388c-4e15-534c-8213-83edad4aa028,FX96-24ms v2 Spot,DZH318Z0H7F2,DZH318Z0H7F2/05V6,Virtual Machines FXmsv2 Series Windows,FX96-24ms v2 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,DevTestConsumption,True,Standard_FX96-24ms_v2,NaN
9,USD,0.0,0.040800,0.040800,centralus,US Central,2018-05-01T00:00:00Z,00834c88-ac6c-4bc9-82fc-34e2e4a51294,vCore,DZH318Z0BQJ3,DZH318Z0BQJ3/007D,Azure Database for PostgreSQL Single Server Ba...,1 vCore,Azure Database for PostgreSQL,DZH3199QPQTD,Databases,1 Hour,Consumption,True,,NaN


Εδώ βλέπουμε μερικά statistics σχετικά με το πλήθος γραμμών και στηλών του πίνακα με της υπηρεσίες. Κάθε υπηρεσία έχει 21 κλειδιά - στοιχεία τα οποία την χαρακτηρίζουν. Παρατηρούμε ότι έχουμε 21 πεδία. Αρκετά λιγότερα αναλογικά με την goolge όπου έν τελη είχαμε 29.

In [43]:
print (f"The array has {df_flat.shape[0]} lines/rows")
print (f"The array had {df_flat.shape[1]} columns")

The array has 1000 lines/rows
The array had 21 columns


Από τα δεδομένα που έχω, ασχέτως ποια θα ανεβάσω αργότερα ως καθαρά στο Minio τα οποία θα αφορούν VM, STORAGE, κλπ, ποιες εγγραφές μπορώ να τις πετάξω ως ελλιπείς. Πχ ποιες έχουν Nan, ή κάτι άλλο; 
- Η πιο απλή σκέψη -> Ψάχνω σε οποιαδήποτε στήλη για οποιαδήποτε εγγραφη Nan εκτός της reservationTerm
    - Ξεκινάω από την στήλη με τον τύπο νομίσματος. Αν υπάρχει Nan το πετάω όλο σαν εγγραφή καθώς δεν είναι valid
- Eπίσης κοιταζω στήλες που δεν έχουν καν τιμή (όχι Nan αλλά πλήρης απουσία τιμής)
Τέλειωσα με τα Nan στο κομμάτι του νομίσματος!! Τι άλλο??
- Το έαν όλες οι τιμές είναι δολάριο ή κάποιο άλλο νόμισμα για την ώρα δεν με ενδιαφέρει, και μάλλον δεν θα μας ενδιαφέρει, καθώς όπως ορίζεται στο Documentation του API, εκτός και αν δηλωθεί ρητά (με query param όταν καλώ το API), όλες οι τιμές θα είναι σε USD (φίλτρο PriceType)

In [44]:
# Η παρακάτω μεταβλητή κρατάει το πλήθος των κενών τιμών από την στήλη με τον τύπο νομίσματος. Η μεταβλητή δεν είναι νεο df, απλή μεταβλητή int
nan_count = df_flat['currencyCode'].isna().sum()
# print (type(nan_count))
print(f"Total Nan values in currencyCode column are: {nan_count}")

# Αν βρούμε έστω και μία Nan τιμή τρέχουμε μπαίνει στο loop για να την πετάξει και επαναφέρει και το index στις άλλες εγγραφές
if nan_count > 0:
    df_flat = df_flat.dropna(subset=['currencyCode']).reset_index(drop=True)
    print(f"New df size: {df_flat.shape[0]} records.")

df_flat.head(10)

Total Nan values in currencyCode column are: 0


,currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName,reservationTerm
0,USD,0.0,0.505452,0.505452,usgovtexas,US Gov TX,2018-11-01T00:00:00Z,0cab4972-6771-43c3-a756-e62e08ae8e75,D2 v2 AHB,DZH318Z0BQN9,DZH318Z0BQN9/004D,SSIS Standard D-series v2 VM,D2 v2,Azure Data Factory v2,DZH315FBNNW2,Analytics,1 Hour,Consumption,False,,NaN
1,USD,0.0,0.000000,0.000000,austriaeast,AT East,2025-05-01T00:00:00Z,ba97cdcf-f753-5722-9a3b-cd0f7130c423,Delete Operations,DZH318Z0BP09,DZH318Z0BP09/01D5,Files v2,Standard LRS,Storage,DZH317F1HKN0,Storage,10K,Consumption,False,,NaN
2,USD,0.0,0.037500,0.037500,uksouth,UK South,2023-05-01T00:00:00Z,0082698b-72e6-5709-8e5c-9e1553f91378,Cold ZRS Index Tags,DZH318Z0BPH7,DZH318Z0BPH7/02JK,General Block Blob v2,Cold ZRS,Storage,DZH317F1HKN0,Storage,10K/Month,Consumption,True,Cold,NaN
3,USD,0.0,0.030000,0.030000,francecentral,FR Central,2021-08-01T00:00:00Z,0a623b39-fbc1-5b4a-a451-bcf16f060fe6,Azure Maps Creator Feature State,DZH318Z0BQ9W,DZH318Z0BQ9W/0034,Azure Maps,Azure Maps Creator,Azure Maps,DZH315B3SJD3,Internet of Things,1K,Consumption,False,,NaN
4,USD,0.0,1.808000,1.808000,westus2,US West 2,2021-11-01T00:00:00Z,0082c735-0257-5cd1-8505-d2b775605910,D32d v5,DZH318Z08MC2,DZH318Z08MC2/004V,Virtual Machines Ddv5 Series,Standard_D32d_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_D32d_v5,NaN
5,USD,0.0,9344.000000,9344.000000,westus2,US West 2,2025-04-01T00:00:00Z,0082c735-0257-5cd1-8505-d2b775605910,D32d v5,DZH318Z08MC2,DZH318Z08MC2/01ZQ,Virtual Machines Ddv5 Series,Standard_D32d_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Reservation,True,Standard_D32d_v5,1 Year
6,USD,0.0,18055.000000,18055.000000,westus2,US West 2,2021-11-01T00:00:00Z,0082c735-0257-5cd1-8505-d2b775605910,D32d v5,DZH318Z08MC2,DZH318Z08MC2/01ZN,Virtual Machines Ddv5 Series,Standard_D32d_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Reservation,True,Standard_D32d_v5,3 Years
7,USD,0.0,3.620417,3.620417,southafricanorth,ZA North,2025-08-01T00:00:00Z,0083388c-4e15-534c-8213-83edad4aa028,FX96-24ms v2 Spot,DZH318Z0H7F2,DZH318Z0H7F2/05V6,Virtual Machines FXmsv2 Series Windows,FX96-24ms v2 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_FX96-24ms_v2,NaN
8,USD,0.0,2.804340,2.804340,southafricanorth,ZA North,2025-08-01T00:00:00Z,0083388c-4e15-534c-8213-83edad4aa028,FX96-24ms v2 Spot,DZH318Z0H7F2,DZH318Z0H7F2/05V6,Virtual Machines FXmsv2 Series Windows,FX96-24ms v2 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,DevTestConsumption,True,Standard_FX96-24ms_v2,NaN
9,USD,0.0,0.040800,0.040800,centralus,US Central,2018-05-01T00:00:00Z,00834c88-ac6c-4bc9-82fc-34e2e4a51294,vCore,DZH318Z0BQJ3,DZH318Z0BQJ3/007D,Azure Database for PostgreSQL Single Server Ba...,1 vCore,Azure Database for PostgreSQL,DZH3199QPQTD,Databases,1 Hour,Consumption,True,,NaN


- Με ενδιαφέρει: αν κάποια εγγραφή έχει τιμή 0; Μάλλον πρέπει να πάει για σουτ
    - Άρα εδώ κοιτάζω τις στήλες retailPrice και unitPrice, και ανάλογα καθαρίζω
Τι είναι το καθένα όμως ; (Και τα δύο σχετίζονται με την τιμολόγηση της υπηρεσίας το μόνο που ξέρουμε με σιγουριά)
- Το retail price αφορά fix τιμή που πληρώνει κάποιος απλός πελάτης 
- Το unit price αφορά τιμή μετά εκτπώσεων... και είναι πάντα μικρότερο ή ίσο του retailPrice. Θα κρατήσουμε μόνο το πρώτο, καθώς θέλω έναν δημόδιο δείκτης αναφοράς σχετικά με την τιμή της υπηρεσίας προκειμένου να προχωρήσω στην σύγκριση με τους άλλους παρόχους, και όχι κάτι ad - hoc το οποίο εξαρτάται από το προφίλ του καταναλωτή. Οι άλλοι 2 πάροχοι δεν δίνουν αντίστοιχο του unitPrice

In [45]:
zero_nan= (df_flat['retailPrice'] <= 0) | (df_flat['retailPrice'].isna())
zero_nan_count = zero_nan.sum()
print (f"From the {len(df_flat)} records, {zero_nan_count} had either 0 in the retail price or Nan")

if zero_nan_count > 0:
    df_flat= df_flat[df_flat['retailPrice'] > 0].reset_index(drop=True)

df_flat= df_flat.drop(columns=['unitPrice'], errors='ignore')

print (f"The array has {df_flat.shape[0]} lines/rows")
print (f"The array had {df_flat.shape[1]} columns")

df_flat.head(10)

From the 1000 records, 27 had either 0 in the retail price or Nan
The array has 973 lines/rows
The array had 20 columns


,currencyCode,tierMinimumUnits,retailPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName,reservationTerm
0,USD,0.0,0.505452,usgovtexas,US Gov TX,2018-11-01T00:00:00Z,0cab4972-6771-43c3-a756-e62e08ae8e75,D2 v2 AHB,DZH318Z0BQN9,DZH318Z0BQN9/004D,SSIS Standard D-series v2 VM,D2 v2,Azure Data Factory v2,DZH315FBNNW2,Analytics,1 Hour,Consumption,False,,NaN
1,USD,0.0,0.037500,uksouth,UK South,2023-05-01T00:00:00Z,0082698b-72e6-5709-8e5c-9e1553f91378,Cold ZRS Index Tags,DZH318Z0BPH7,DZH318Z0BPH7/02JK,General Block Blob v2,Cold ZRS,Storage,DZH317F1HKN0,Storage,10K/Month,Consumption,True,Cold,NaN
2,USD,0.0,0.030000,francecentral,FR Central,2021-08-01T00:00:00Z,0a623b39-fbc1-5b4a-a451-bcf16f060fe6,Azure Maps Creator Feature State,DZH318Z0BQ9W,DZH318Z0BQ9W/0034,Azure Maps,Azure Maps Creator,Azure Maps,DZH315B3SJD3,Internet of Things,1K,Consumption,False,,NaN
3,USD,0.0,1.808000,westus2,US West 2,2021-11-01T00:00:00Z,0082c735-0257-5cd1-8505-d2b775605910,D32d v5,DZH318Z08MC2,DZH318Z08MC2/004V,Virtual Machines Ddv5 Series,Standard_D32d_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_D32d_v5,NaN
4,USD,0.0,9344.000000,westus2,US West 2,2025-04-01T00:00:00Z,0082c735-0257-5cd1-8505-d2b775605910,D32d v5,DZH318Z08MC2,DZH318Z08MC2/01ZQ,Virtual Machines Ddv5 Series,Standard_D32d_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Reservation,True,Standard_D32d_v5,1 Year
5,USD,0.0,18055.000000,westus2,US West 2,2021-11-01T00:00:00Z,0082c735-0257-5cd1-8505-d2b775605910,D32d v5,DZH318Z08MC2,DZH318Z08MC2/01ZN,Virtual Machines Ddv5 Series,Standard_D32d_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Reservation,True,Standard_D32d_v5,3 Years
6,USD,0.0,3.620417,southafricanorth,ZA North,2025-08-01T00:00:00Z,0083388c-4e15-534c-8213-83edad4aa028,FX96-24ms v2 Spot,DZH318Z0H7F2,DZH318Z0H7F2/05V6,Virtual Machines FXmsv2 Series Windows,FX96-24ms v2 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_FX96-24ms_v2,NaN
7,USD,0.0,2.804340,southafricanorth,ZA North,2025-08-01T00:00:00Z,0083388c-4e15-534c-8213-83edad4aa028,FX96-24ms v2 Spot,DZH318Z0H7F2,DZH318Z0H7F2/05V6,Virtual Machines FXmsv2 Series Windows,FX96-24ms v2 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,DevTestConsumption,True,Standard_FX96-24ms_v2,NaN
8,USD,0.0,0.040800,centralus,US Central,2018-05-01T00:00:00Z,00834c88-ac6c-4bc9-82fc-34e2e4a51294,vCore,DZH318Z0BQJ3,DZH318Z0BQJ3/007D,Azure Database for PostgreSQL Single Server Ba...,1 vCore,Azure Database for PostgreSQL,DZH3199QPQTD,Databases,1 Hour,Consumption,True,,NaN
9,USD,0.0,1.480000,centralus,US Central,2023-11-01T00:00:00Z,00835171-caea-553f-8e12-c30b9bfae643,EC20adscc v5,DZH318Z0BDCH,DZH318Z0BDCH/00DL,Virtual Machines ECadsccv5 Series,Standard_EC20ads_cc_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_EC20ads_cc_v5,NaN


Πλέον όλες οι εγγραφές είναι σε δολάριο (η στήλη του νομίσματος δεν έχει Nan), και η λιανική τιμή >0
- Βάση της παραδοχής που έγινε ενδιαφερόμαστε μόνο για pay as you go υπηρεσίες ή αλλιώς consumption και όχι υπηρεσίες που έχουν ειδικές τιμές μέσω μακροχρόνιων συμβολαίων. Άρα εφαρμόζω φίλτρο με το οποίο πετάω όσα δεν με καλύτπουν, καθώς και ολόκληρη την στήλη reservationTerm, αφού αφορά έτη μακροχρόνιων συμβολαίων.

In [46]:
df_flat = df_flat[df_flat['type'] == 'Consumption']
df_flat = df_flat.reset_index(drop=True)
df_flat= df_flat.drop(columns=['reservationTerm'], errors='ignore')

print (f"The array has {df_flat.shape[0]} lines/rows")
print (f"The array had {df_flat.shape[1]} columns")

df_flat.head(10)

The array has 661 lines/rows
The array had 19 columns


,currencyCode,tierMinimumUnits,retailPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName
0,USD,0.0,0.505452,usgovtexas,US Gov TX,2018-11-01T00:00:00Z,0cab4972-6771-43c3-a756-e62e08ae8e75,D2 v2 AHB,DZH318Z0BQN9,DZH318Z0BQN9/004D,SSIS Standard D-series v2 VM,D2 v2,Azure Data Factory v2,DZH315FBNNW2,Analytics,1 Hour,Consumption,False,
1,USD,0.0,0.037500,uksouth,UK South,2023-05-01T00:00:00Z,0082698b-72e6-5709-8e5c-9e1553f91378,Cold ZRS Index Tags,DZH318Z0BPH7,DZH318Z0BPH7/02JK,General Block Blob v2,Cold ZRS,Storage,DZH317F1HKN0,Storage,10K/Month,Consumption,True,Cold
2,USD,0.0,0.030000,francecentral,FR Central,2021-08-01T00:00:00Z,0a623b39-fbc1-5b4a-a451-bcf16f060fe6,Azure Maps Creator Feature State,DZH318Z0BQ9W,DZH318Z0BQ9W/0034,Azure Maps,Azure Maps Creator,Azure Maps,DZH315B3SJD3,Internet of Things,1K,Consumption,False,
3,USD,0.0,1.808000,westus2,US West 2,2021-11-01T00:00:00Z,0082c735-0257-5cd1-8505-d2b775605910,D32d v5,DZH318Z08MC2,DZH318Z08MC2/004V,Virtual Machines Ddv5 Series,Standard_D32d_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_D32d_v5
4,USD,0.0,3.620417,southafricanorth,ZA North,2025-08-01T00:00:00Z,0083388c-4e15-534c-8213-83edad4aa028,FX96-24ms v2 Spot,DZH318Z0H7F2,DZH318Z0H7F2/05V6,Virtual Machines FXmsv2 Series Windows,FX96-24ms v2 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_FX96-24ms_v2
5,USD,0.0,0.040800,centralus,US Central,2018-05-01T00:00:00Z,00834c88-ac6c-4bc9-82fc-34e2e4a51294,vCore,DZH318Z0BQJ3,DZH318Z0BQJ3/007D,Azure Database for PostgreSQL Single Server Ba...,1 vCore,Azure Database for PostgreSQL,DZH3199QPQTD,Databases,1 Hour,Consumption,True,
6,USD,0.0,1.480000,centralus,US Central,2023-11-01T00:00:00Z,00835171-caea-553f-8e12-c30b9bfae643,EC20adscc v5,DZH318Z0BDCH,DZH318Z0BDCH/00DL,Virtual Machines ECadsccv5 Series,Standard_EC20ads_cc_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_EC20ads_cc_v5
7,USD,0.0,15.000000,eastus,US East,2016-10-01T00:00:00Z,a4e29a95-5b4c-408b-80e3-113f9410566e,Standard Node,DZH318Z0BQV2,DZH318Z0BQV2/0006,Insight and Analytics,Standard,Insight and Analytics,DZH318Z30KPP,Management and Governance,1/Month,Consumption,False,
8,USD,0.0,27.589000,indiasouthcentral,IN South Central,2026-05-01T00:00:00Z,549fa486-6b94-549b-b5ea-48b6b9bc4a68,vCore,DZH318Z0B172,DZH318Z0B172/03W5,SQL Managed Instance General Purpose - Premium...,80 vCore,SQL Managed Instance,DZH318XWNZTN,Databases,1 Hour,Consumption,False,80 vCore
9,USD,0.0,2.046845,swedensouth,SE South,2025-10-01T00:00:00Z,00837e0f-2c93-59c8-9fca-1ca7a9bfa38d,E96bds v5 Spot,DZH318Z09LQP,DZH318Z09LQP/02FQ,Virtual Machines Ebdsv5 Series,Standard_E96bds_v5 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_E96bds_v5


(Όλες οι τιμές είναι σε δολάριο χωρίς Ναn ή τιμή μιρκότερη του μηδενός)
- Έχω πολλές στήλες με ονόματα. Τις θέλω όλες ; Τι συμβολίζει η κάθε μία ; (meterName, skuName, armSkuName, serviceName, productName)
- Ποια η διαφορά armRegion με location ;
- Γενικά ότι έχει μπροστά "arm" τι είναι ;
- Ποια η διαφορά service, product και sku;
- Με ενδιαφέρουν οι στήλες armSkuName και reservationTerm. H πρώτη φαίνεται σε πολλά να μην έχει τιμές καν, ενώ η 2η έχει αρκετά Nan.


Από εδώ και στο εξής, πρέπει να σκεφτούμε την κατηγοριοποίηση που θα κάνουμε, και να απαντήσουμε σε βασικά ερωτήματα τα οποία αφορούν τα δεδομένα
1. Σίγουρα δεν τα θέλω όλα τα δεδομένα!!! Ποια θέλω να κρατήσω ?
2. Πιθανά κριτήρια φιλτραρίσματος είναι το serviceName (Virtual Machines, Storage) και το serviceFamily (Compute,Storage).  Ποιο από τα δύο ?
3. Ποια η διαφορά retail price και unitPrice? Και τα δύο φαίνονται ίδια. Ισχύει ?
4. Τι ισχύει με το type? Υπάρχουν 3 διαφορετικοί τύποι και ανάλογα διαφορετικές χρεώσεις με μεγάλες διαφορές. (Reservatiom, Consumption, DevTestConsumption) Τι είναι το καθένα?
    - Στο reservation έχω συγκριτικά πολύ μεγάλες τιμές
    - Επίσης όταν έχω reservation στον τύπο η στήλη reservationTerm έχει πάντα τιμή σε περίοδο χρόνου (πχ ν χρόνια), ενώ στις άλλες περιπτώσεις δεν έχω καν τιμή (NaN)

**Μεταγενέστετο αλλά κομβικό**
- Στην google αυτό το οποίο παρέδωσα είναι 29 στήλες. Εδώ γιατί 22? Τι διαφορές έχω ; Τι λείπει και τι παραπάνω έχουν οι άλλοι ?
- Από τις στήλες που έχω εδώ μου χρειάζονται όλες? Μπορώ να πετάξω κάτι ? Το ίδιο πρέπει να ανατωτηθώ και στην google αλλά ίσως η απάντηση και το επιπλέον καθάρισμα των δεδομένων να δοθεί όταν θα φτιάχνω τι εννιαίο σχήμα

**Πως το σκέφτομαι**
- Φιλτράρισμα βάση serviceName, αποθήκευση και μετά επιπλέον φιλτράρισμα βάση τύπου (type)
- Θα έχω δηλαδή : vm_reservation.json, vm_consumption.json, ....
- Πόσες υπηρεσίες όμως υπάρχουν για κάθε κατηγορία που θέλω;
    - Για την ώρα έχω 1000 json από 1000 services το καθένα. Θα τα διαβάζω όλα και θα κρατάω κάθε φορά αυτό που θέλω και θα συνθέτω;
    - Θα κάνω ένα πρώτο στάδιο με 100 σελίδες (να βγάλω 10 δηλαδή) και μετά επιπλέον concat?